# Project 1 · CIFAR-10 猫狗图像分类

我从 CIFAR-10 中筛选猫和狗，先手写 CNN 和训练循环，再依次尝试数据增强、权重衰减、改进结构、学习率调度，以及预训练 ResNet18。

**任务：** 二分类，`0 = cat`、`1 = dog`。输入原图为 `[3, 32, 32]`，不是 CIFAR-10 的完整十分类任务。

**阅读方式：** 按顺序运行。首次需要联网下载数据和 ResNet18 权重。文末表格来自原文件的历史输出；整理版清除了旧执行输出，尚未完整重训。

## 环境

建议使用 Python 3.10+，先安装同目录的 `requirements.txt`。有 CUDA 时自动使用 GPU，否则使用 CPU。所有模型和批次会放到同一设备；CPU 完整运行全部实验可能较慢。

In [ ]:
import random
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)

In [ ]:
import matplotlib.pyplot as plt

## 1. 读取数据

CIFAR-10 的原始训练集有 50,000 张 32×32 彩色图片，包含 10 类。本项目只取其中的猫和狗，各 5,000 张。官方测试集在本实验中尚未使用。

In [ ]:
train_data = datasets.CIFAR10(root = "./data" , train = True , download = True)

查看原始类别列表，确认猫和狗的编号分别是 `3` 和 `5`。

In [ ]:
train_data.classes

`train_data` 是 Dataset 对象；`train_data[i]` 返回 `(image, label)`，其中图像是 PIL Image，标签是类别编号。

In [ ]:
image,label = train_data[0]

In [ ]:
print("原始类别编号:", label, "| 类别:", train_data.classes[label])

In [ ]:
#用matplotlib 来show image来看image是怎样的
plt.imshow(image)
plt.title(train_data.classes[label])
plt.axis("off")
plt.show()

## 2. 筛选猫狗并转换标签

自定义 Dataset 将原标签 `3 / 5` 映射成 `0 / 1`，并把图像转换为 Tensor。筛选后共有 10,000 张图片，每张形状为 `[3, 32, 32]`，三个通道分别是红、绿、蓝。

In [ ]:
#cat_indices,dog_indices 来分别记录他们在train_data 里面 dog和cat在什么位置
cat_indices = []
dog_indices = []
for i, label in enumerate(train_data.targets):
  if label == 3:
    cat_indices.append(i)
  if label == 5:
    dog_indices.append(i)

In [ ]:
from torch.utils.data import Dataset
from torchvision import transforms

class CatDogDataset(Dataset):
    def __init__(self, data, indices):
        self.data = data
        self.indices = indices
        self.transform = transforms.ToTensor()
        self.label_map = {3: 0, 5: 1}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, original_label = self.data[self.indices[idx]]
        # ToTensor: [H, W, C] 的像素转为 [C, H, W]、范围 [0, 1] 的浮点 Tensor。
        image = self.transform(image)
        label = self.label_map[original_label]
        return image, label

In [ ]:
indices = cat_indices + dog_indices
dataset = CatDogDataset(train_data, indices)
image, label = dataset[0]
print("猫狗样本数:", len(dataset))
print("图像形状:", image.shape, "| 标签:", label)

## 3. 划分数据与批次

使用固定种子将猫狗数据随机分成 8,000 张训练、2,000 张验证，二者索引不重叠。此处是随机划分，不保证每部分猫狗数量完全相同。

DataLoader 每批读取 32 张图片；只在训练时打乱顺序。每个实验重置模型种子和训练 loader 的生成器，减少随机初始化与样本顺序对比较的影响。不同设备与依赖版本仍可能产生不同结果。

In [ ]:
from torch.utils.data import random_split
total_size = len(dataset)
train_size = int(0.8*total_size)
val_size = total_size - train_size
train_dataset , val_dataset = random_split(
    dataset,
    [train_size,val_size],
     generator=torch.Generator().manual_seed(42)
)

In [ ]:
from torch.utils.data import DataLoader

# 各实验开始前重置训练 loader 的随机生成器，固定样本打乱顺序。
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED)
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

In [ ]:
images, labels = next(iter(train_loader))
print("Images:", images.shape)
print("Labels:", labels.shape)

## 4. 手写 CNN 基线

两组卷积、ReLU 和最大池化，依次把形状从 `[B, 3, 32, 32]` 变为 `[B, 16, 16, 16]`、`[B, 32, 8, 8]`。卷积学习局部特征；ReLU 引入非线性；池化缩小空间尺寸。

随后把每张图像的特征展平为 2,048 维，再通过全连接层输出两个 logits。`argmax` 选择分数较高的类别。

In [ ]:
import torch.nn as nn

class CatDogCNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.features = nn.Sequential(
          nn.Conv2d(3,16,kernel_size = 3 , padding = 1),
          #输入3个channel,输出16个特征图，计算新的特征
          nn.ReLU(),
          #max(0,x)
          nn.MaxPool2d(2),
          #保留局部最大值也就是2*2 ，缩小特征图 所以就是 把32*32变成16*16

          nn.Conv2d(16,32,kernel_size = 3 , padding = 1),
          nn.ReLU(),
          nn.MaxPool2d(2)
      )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        #把每张32*8*8 的特征图拉平变成一条长向量
        nn.Linear(32*8*8,64),
        #把每张图片的2048个输入值，通过可学习的权重转换成64个输出数值
        nn.ReLU(),
        nn.Linear(64,2)
        #再把他们输出变成2个数值
    )
  def forward(self,x):
    x = self.features(x)
    x = self.classifier(x)
    return x

### 损失、训练与模型选择

一个批次的流程是：清空梯度 → 前向计算 → 交叉熵损失 → 反向传播 → 优化器更新参数。

验证阶段使用 `eval()` 与 `inference_mode()`，不更新参数。按样本数加权统计平均损失，并保存验证损失最低的权重；连续 3 轮没有改善就提前停止。最后恢复最佳权重，而不是直接使用最后一轮的模型。

In [ ]:
import torch.optim as optim

set_seed()
train_loader.generator.manual_seed(SEED)
model = CatDogCNN().to(device)
criterion = nn.CrossEntropyLoss()
# 直接输入两个类别的 logits；不需要先手动做 softmax。
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
#开始训练
best_val_loss = float("inf")
best_accuracy = 0.0
best_epoch = 0
patience = 3
counter = 0

for epoch in range(15):

    # 1. Training
    model.train()
    train_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        #把全部grad 变成0
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)

    train_loss /= len(train_dataset)

    # 2. Validation
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    val_loss /= total
    accuracy = correct / total

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Accuracy: {accuracy:.2%}"
    )

    # 3. Model Checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_accuracy = accuracy
        best_epoch = epoch + 1
        counter = 0

        torch.save(model.state_dict(), "best_model.pth")
        print("Best model saved!")

    else:
        counter += 1

    # 4. Early Stopping 不希望一直没有进展
    if counter >= patience:
        print("Early stopping triggered!")
        break

# Restore the best saved weights
model.load_state_dict(
    torch.load(
        "best_model.pth",
        map_location=device,
        weights_only=True
    )
)
model.eval()

print("Best Validation Loss:", best_val_loss)

print("Best Epoch:", best_epoch)
print("Accuracy at Best Val Loss:", best_accuracy)

## 5. 实验一：数据增强

仅对训练集使用随机裁剪和水平翻转，增加输入变化；验证集保持固定。增强有助于减少对训练图像细节的依赖，但不保证每次实验都会提高准确率。

下面保留原实验顺序：数据增强 → 加入权重衰减 → 改进 CNN → 学习率调度。

In [ ]:
from torchvision import transforms
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    #把图片上下左右增加4个边框 变成32+4+4 = 40*40 然后再拿来crop 32*32
    transforms.RandomHorizontalFlip(p=0.5) #有50%的概率水平反转
])

In [ ]:
# 可以看看 original 和augmented后的分别
image, label = train_dataset[0]

fig, axes = plt.subplots(1, 4, figsize=(12, 3))

# Original
axes[0].imshow(image.permute(1, 2, 0))
axes[0].set_title("Original")

# Three augmented versions
for i in range(1, 4):
    augmented = train_transform(image)

    axes[i].imshow(augmented.permute(1, 2, 0))
    axes[i].set_title(f"Augmented {i}")

for ax in axes:
    ax.axis("off")

plt.show()

In [ ]:
# 只对训练集做随机增强，验证集保持固定。
class AugmentedDataset(Dataset):
    def __init__(self, original_dataset, transform):
        self.original_dataset = original_dataset
        self.transform = transform

    def __len__(self):
        return len(self.original_dataset)

    def __getitem__(self, idx):
        image, label = self.original_dataset[idx]
        image = self.transform(image)
        return image, label

aug_train_dataset = AugmentedDataset(
    train_dataset,
    train_transform
)

aug_train_loader = DataLoader(
    aug_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED)
)

In [ ]:
#训练一样只是名字不一样
set_seed()
aug_train_loader.generator.manual_seed(SEED)

aug_model = CatDogCNN().to(device)

criterion = nn.CrossEntropyLoss()
aug_optimizer = torch.optim.Adam(
    aug_model.parameters(), lr=0.001
)

best_val_loss = float("inf")
best_accuracy = 0
best_epoch = 0
patience = 3
counter = 0

for epoch in range(15):

    # Training
    aug_model.train()
    train_loss = 0

    for images, labels in aug_train_loader:
        images = images.to(device)
        labels = labels.to(device)

        aug_optimizer.zero_grad()

        outputs = aug_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        aug_optimizer.step()

        train_loss += loss.item() * labels.size(0)

    train_loss /= len(aug_train_dataset)

    # Validation (without augmentation)
    aug_model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = aug_model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            total += labels.size(0)

    val_loss /= total
    accuracy = correct / total

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Accuracy: {accuracy:.2%}"
    )

    # Early Stopping and Checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_accuracy = accuracy
        best_epoch = epoch + 1
        counter = 0

        torch.save(
            aug_model.state_dict(),
            "best_model_aug.pth"
        )
        print("Best model saved!")
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered!")
        break

# Restore best model
aug_model.load_state_dict(
    torch.load(
        "best_model_aug.pth",
        map_location=device,
        weights_only=True
    )
)

print("Best Epoch:", best_epoch)
print("Best Val Loss:", best_val_loss)
print("Accuracy at Best Val Loss:", best_accuracy)

aug_model.eval()

## 6. 实验二：增强 + 权重衰减

在数据增强基础上，给 Adam 设置 `weight_decay=1e-4`，对参数大小施加惩罚，尝试减少过拟合。这里使用的是 Adam 的 L2 惩罚形式，与 AdamW 的解耦权重衰减不同。

原文件的这段日志只记录到第 14 轮，没有完整结束信息，因此文末不将它作为完整实验成绩。

In [ ]:
#实验2
import torch
import torch.nn as nn
set_seed()
aug_train_loader.generator.manual_seed(SEED)
wd_model = CatDogCNN().to(device)

criterion = nn.CrossEntropyLoss()
wd_optimizer = torch.optim.Adam(
    wd_model.parameters(),
    lr = 0.001,
    weight_decay = 1e-4
)
# Adam 的 L2 惩罚形式；交叉熵的打印值本身不包含这项惩罚。

In [ ]:
best_val_loss = float("inf")
best_accuracy = 0
best_epoch = 0

patience = 3
counter = 0

for epoch in range(15):

    # Training
    wd_model.train()
    train_loss = 0

    for images, labels in aug_train_loader:
        images = images.to(device)
        labels = labels.to(device)

        wd_optimizer.zero_grad()

        outputs = wd_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        wd_optimizer.step()

        train_loss += loss.item() * labels.size(0)

    train_loss /= len(aug_train_dataset)

    # Validation
    wd_model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = wd_model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    val_loss /= total
    accuracy = correct / total

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Accuracy: {accuracy:.2%}"
    )

    # Checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_accuracy = accuracy
        best_epoch = epoch + 1
        counter = 0

        torch.save(
            wd_model.state_dict(),
            "best_model_wd.pth"
        )
        print("Best model saved!")

    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered!")
        break

# Load best model
wd_model.load_state_dict(
    torch.load(
        "best_model_wd.pth",
        map_location=device,
        weights_only=True
    )
)

wd_model.eval()

print("Best Epoch:", best_epoch)
print("Best Val Loss:", best_val_loss)
print("Accuracy at Best Val Loss:", best_accuracy)

## 7. 实验三：改进 CNN

增加第三个卷积块、更多通道和 BatchNorm。`AdaptiveAvgPool2d((1, 1))` 将**每个通道的空间尺寸**压成 `1×1`，保持 128 个通道：`[B, 128, 4, 4] → [B, 128, 1, 1]`。

这次同时改变了多项结构，不能把结果提升单独归因于某一项。该实验保留数据增强，`weight_decay=0`，不是继续叠加上一实验的权重衰减。

In [ ]:
#实验3
import torch
import torch.nn as nn

class ImprovedCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), #加了batchnorm
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            #加了多一层

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), # [B, 128, 4, 4] -> [B, 128, 1, 1]；通道数仍为 128。
            nn.Flatten(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
set_seed()
aug_train_loader.generator.manual_seed(SEED)

improved_model = ImprovedCNN().to(device)
criterion = nn.CrossEntropyLoss()
improved_optimizer = torch.optim.Adam(
    improved_model.parameters(),
    lr=0.001,
    weight_decay=0
)

In [ ]:
best_val_loss = float("inf")
best_accuracy = 0
best_epoch = 0
patience = 4
counter = 0

for epoch in range(15):

    # Training
    improved_model.train()
    train_loss = 0

    for images, labels in aug_train_loader:
        images = images.to(device)
        labels = labels.to(device)

        improved_optimizer.zero_grad()

        outputs = improved_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        improved_optimizer.step()

        train_loss += loss.item() * labels.size(0)

    train_loss /= len(aug_train_dataset)

    # Validation
    improved_model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = improved_model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            total += labels.size(0)

    val_loss /= total
    accuracy = correct / total

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Accuracy: {accuracy:.2%}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_accuracy = accuracy
        best_epoch = epoch + 1
        counter = 0

        torch.save(
            improved_model.state_dict(),
            "best_model_improved.pth"
        )
        print("Best model saved!")
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered!")
        break

# Restore best weights
improved_model.load_state_dict(
    torch.load(
        "best_model_improved.pth",
        map_location=device,
        weights_only=True
    )
)

improved_model.eval()

print("Best Epoch:", best_epoch)
print("Best Val Loss:", best_val_loss)
print("Accuracy at Best Val Loss:", best_accuracy)

## 8. 实验四：学习率调度

继续使用 ImprovedCNN 和数据增强，不使用权重衰减。`StepLR(step_size=5, gamma=0.5)` 在每轮优化后调用，使下一轮开始时的学习率按计划降低：第 1–5 轮 `0.001`，第 6–10 轮 `0.0005`，第 11–15 轮 `0.00025`。

学习率变小不保证最终结果一定超过固定学习率，需要看验证表现。

In [ ]:
#实验4
set_seed()
aug_train_loader.generator.manual_seed(SEED)

lr_model = ImprovedCNN().to(device)

criterion = nn.CrossEntropyLoss()

lr_optimizer = torch.optim.Adam(
    lr_model.parameters(),
    lr=0.001,
    weight_decay=0
)

scheduler = torch.optim.lr_scheduler.StepLR(
    lr_optimizer,
    step_size=5,
    gamma=0.5
)

In [ ]:
best_val_loss = float("inf")
best_accuracy = 0
best_epoch = 0

patience = 4
counter = 0

for epoch in range(15):

    current_lr = lr_optimizer.param_groups[0]["lr"]

    # Training
    lr_model.train()
    train_loss = 0

    for images, labels in aug_train_loader:
        images = images.to(device)
        labels = labels.to(device)

        lr_optimizer.zero_grad()

        outputs = lr_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        lr_optimizer.step()

        train_loss += loss.item() * labels.size(0)

    train_loss /= len(aug_train_dataset)

    # Validation
    lr_model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = lr_model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    val_loss /= total
    accuracy = correct / total

    print(
        f"Epoch {epoch+1} | "
        f"LR: {current_lr:.6f} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Accuracy: {accuracy:.2%}"
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_accuracy = accuracy
        best_epoch = epoch + 1
        counter = 0

        torch.save(
            lr_model.state_dict(),
            "best_model_lr.pth"
        )
        print("Best model saved!")
    else:
        counter += 1

    # Update learning rate for next epoch
    scheduler.step()

    # Early stopping
    if counter >= patience:
        print("Early stopping triggered!")
        break

# Restore best checkpoint
lr_model.load_state_dict(
    torch.load(
        "best_model_lr.pth",
        map_location=device,
        weights_only=True
    )
)

lr_model.eval()

print("Best Epoch:", best_epoch)
print("Best Val Loss:", best_val_loss)
print("Accuracy at Best Val Loss:", best_accuracy)

## 9. 实验五：ResNet18 迁移学习

使用 ImageNet 预训练 ResNet18，冻结主干参数，只训练新的二分类层。这里是**固定特征提取器的迁移学习**，尚未解冻主干进行 fine-tuning。

预处理使用对应权重提供的缩放、中心裁剪与归一化；训练集额外加入随机增强。放大 32×32 原图只是适配预训练输入，不会创造新的图像细节。

训练时保持主干为 `eval()`，防止 BatchNorm 运行统计量发生变化；新的 `Linear(512, 2)` 有 1,026 个可训练参数。验证损失最低的一轮才是最终保存的模型。

In [ ]:
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from tqdm.auto import tqdm


# ==================================================
# 1. 随机种子和设备
# ==================================================
set_seed()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("当前使用 CPU；在 Colab 启用 GPU 后可以加快训练。")


# ==================================================
# 2. 预处理
# 前提：之前的 dataset 返回 [0, 1] 的图片 Tensor，
# 标签为 0=cat、1=dog。
# ==================================================
weights = ResNet18_Weights.DEFAULT

resnet_train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    weights.transforms()
])

# 验证集不使用随机增强
resnet_val_transform = weights.transforms()


class ResNetDataset(Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        image = self.transform(image)
        return image, label


resnet_train_dataset = ResNetDataset(
    train_dataset,
    resnet_train_transform
)

resnet_val_dataset = ResNetDataset(
    val_dataset,
    resnet_val_transform
)


# ==================================================
# 3. DataLoader
# ==================================================
use_cuda = device.type == "cuda"

# 默认单进程读取，便于在不同操作系统的 notebook 中运行。
# 确认环境支持后，可以再增加 num_workers。
num_workers = 0

resnet_train_loader = DataLoader(
    resnet_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=use_cuda,
    persistent_workers=(num_workers > 0),
    generator=torch.Generator().manual_seed(42)
)

resnet_val_loader = DataLoader(
    resnet_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=use_cuda,
    persistent_workers=(num_workers > 0)
)


# ==================================================
# 4. 创建预训练模型
# 每次运行这段代码，都会重新创建模型
# ==================================================
resnet_model = resnet18(weights=weights)

# 冻结原来的全部参数
for param in resnet_model.parameters():
    param.requires_grad = False

# 替换分类层
# 新建 Linear 的参数默认允许训练
num_features = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_features, 2)

# 先移动模型，再创建 optimizer
resnet_model = resnet_model.to(device)

print("Model device:", next(resnet_model.parameters()).device)

trainable_params = sum(
    p.numel()
    for p in resnet_model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable_params)


# ==================================================
# 5. 损失函数和优化器
# ==================================================
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    resnet_model.fc.parameters(),
    lr=0.001
)


# ==================================================
# 6. 开始训练
# ==================================================
num_epochs = 5

best_val_loss = float("inf")
best_accuracy = 0.0
best_epoch = 0

checkpoint_path = "best_resnet18.pth"

for epoch in range(num_epochs):

    # 保持预训练 BatchNorm 的统计量不变
    resnet_model.eval()

    # 只训练新的分类层
    # Linear 在 train/eval 下计算相同，
    # 这里显式标明它处于训练状态
    resnet_model.fc.train()

    train_loss_sum = 0.0
    train_total = 0

    train_bar = tqdm(
        resnet_train_loader,
        desc=f"Epoch {epoch + 1}/{num_epochs} | Train"
    )

    for images, labels in train_bar:

        # 模型、图片、标签必须在同一个设备上
        images = images.to(
            device, non_blocking=use_cuda
        )
        labels = labels.to(
            device, non_blocking=use_cuda
        )

        optimizer.zero_grad(set_to_none=True)

        outputs = resnet_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        train_loss_sum += loss.item() * batch_size
        train_total += batch_size

        train_bar.set_postfix(
            loss=f"{train_loss_sum / train_total:.4f}"
        )

    train_loss = train_loss_sum / train_total

    # ----------------------------------------------
    # Validation
    # ----------------------------------------------
    resnet_model.eval()

    val_loss_sum = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():

        for images, labels in tqdm(
            resnet_val_loader,
            desc=f"Epoch {epoch + 1}/{num_epochs} | Val"
        ):
            images = images.to(
                device, non_blocking=use_cuda
            )
            labels = labels.to(
                device, non_blocking=use_cuda
            )

            outputs = resnet_model(images)
            loss = criterion(outputs, labels)

            batch_size = labels.size(0)

            val_loss_sum += loss.item() * batch_size

            predictions = outputs.argmax(dim=1)

            correct += (
                predictions == labels
            ).sum().item()

            total += batch_size

    val_loss = val_loss_sum / total
    accuracy = correct / total

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {accuracy:.2%}"
    )

    # 按最低验证 loss 保存模型
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_accuracy = accuracy
        best_epoch = epoch + 1

        torch.save(
            resnet_model.state_dict(),
            checkpoint_path
        )

        print("Best model saved!")


# ==================================================
# 7. 恢复最佳模型
# ==================================================
resnet_model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
)

resnet_model.eval()

print()
print("Best Epoch:", best_epoch)
print(f"Best Val Loss: {best_val_loss:.4f}")
print(f"Accuracy at Best Val Loss: {best_accuracy:.2%}")
print("Model Device:", next(resnet_model.parameters()).device)

## 10. 结果与我的理解

以下为**原 notebook 的历史记录**，不是整理版重训结果。表中准确率统一取验证损失最低的那一轮，不是各实验出现过的最高准确率。

| 实验 | 选择的轮数 | 验证损失 | 该轮验证准确率 |
| --- | --- | --- | --- |
| 基础 CNN | 10 | 0.5468 | 72.65% |
| CNN + 数据增强 | 14 | 0.4873 | 75.40% |
| CNN + 数据增强 + 权重衰减 | 日志不完整 | — | — |
| ImprovedCNN + 数据增强 | 15 | 0.4139 | 80.90% |
| ImprovedCNN + 数据增强 + StepLR | 13 | 0.4145 | 80.30% |
| 预训练 ResNet18，只训练分类层 | 4 | 0.2851 | 87.70% |

- 我从头训练了 CNN，并观察训练损失、验证损失和准确率如何变化。
- 数据增强、正则化和学习率调度都有各自作用，需要用验证结果判断；更复杂的方法未必每次更好。
- 迁移学习可以复用预训练特征，但这里尚未做主干微调；下一步可以解冻部分层，用较小学习率训练。

**局限：** 目前是猫狗二分类；反复使用同一验证集选择方案，会使验证成绩偏乐观。不同模型的初始化、预处理和训练预算也不完全相同，不能把这张表当作严格的单变量消融实验。官方独立测试集尚未评估，完整原始日志见 `results/original_training_logs.txt`。

**整理修正：** 统一了设备处理、实验种子与 checkpoint 加载，补充基线最佳轮数记录，修正 Dataset、池化及迁移学习的说明，并移除重复模型加载。模型结构、数据划分种子、优化器参数、训练轮数及停止规则沿用原实验设置。新运行结果可能不同。

## 参考资料

- [CIFAR-10 官方数据集](https://www.cs.toronto.edu/~kriz/cifar.html)
- [PyTorch 迁移学习教程](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [torchvision ResNet18 与预训练权重](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)